In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Fan-out/fan-in, saga compensation, bounded reflection, budgets

The research pipeline in `lra.examples` uses all of the patterns together. In this notebook, we run the pipeline, break it and look at how it recovers.

In [ ]:
from datetime import timedelta
import json
from lra import Engine, Event, Budget, Workflow, Next, Done, Wait, StepFailed, RunStatus, StepTask, SimulatedCrash
from lra.adapters.memory import FakeClock, FakeLLM, InMemoryStateStore, InMemoryTaskQueue, InMemoryEventBus, LocalRunner
from lra.examples import ALL_WORKFLOWS
from lra.examples.scripted import research_routes

def harness(routes=None, fail_times=0, chaos=None, lease_ttl_s=60):
    """One engine over in-memory adapters, driven the way Cloud Tasks would drive it."""
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    llm = FakeLLM(routes=routes or research_routes(), fail_times=fail_times)
    engine = Engine(store=store, queue=queue, bus=bus, llm=llm, clock=clock, workflows=ALL_WORKFLOWS,
                    worker_id="worker-A", lease_ttl=timedelta(seconds=lease_ttl_s), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store, queue, bus

def show(run):
    print(f"{run.run_id} {run.status.value:12s} step={run.current_step} attempts={run.attempts} "
          f"wait={run.wait.key if run.wait else None} steps_used={run.budget.steps_used}")

## Orchestrator–worker: a plan becomes child runs

In [ ]:
# TODO: fill in the blanks: wait_kind
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: wait_kind")
engine, runner, clock, store, queue, bus = harness()
run = engine.start("research_pipeline", {"goal": "Why do long-running agents need explicit state?"})
runner.step()                                   # plan -> FanOut
parent = store.get(run.run_id); show(parent)
print("fan_in:", parent.fan_in.model_dump())
children = [c for c in store.list_runs() if c.parent_run_id == run.run_id]
print("children:", [(c.run_id, c.status.value, c.budget.max_cost_usd) for c in children])
assert parent.wait.kind == ____ and len(children) == 3
runner.run_until_idle()
parent = store.get(run.run_id); show(parent)
print("child results:", list(parent.state["children"]["results"]))
print("reflection:", parent.state["reflect_loop"]["exit_reason"])

The children are **runs**. Each child has its own id (`parent--childkey`), budget, retries and history. Because of this id, a second spawn of the children after a crash is idempotent. The parent waited while the children ran.

## Partial failure is data

The source of one subtopic is permanently down. The child fails after its 3 attempts. The aggregator decides that 2/3 is sufficient.

In [ ]:
# TODO: fill in the blanks: child_attempts
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: child_attempts")
routes = research_routes()
def flaky(prompt):
    if "Subtopic 2" in prompt:
        raise RuntimeError("source unavailable")
    return "- finding"
routes[r"Research this subtopic"] = flaky
engine, runner, clock, store, queue, bus = harness(routes=routes)
run = engine.start("research_pipeline", {"goal": "partial"}); runner.run_until_idle()
parent = store.get(run.run_id); show(parent)
print("failures:", parent.state["partial_failures"])
failed = [c for c in store.list_runs() if c.parent_run_id == run.run_id and c.status == RunStatus.FAILED]
assert parent.status == RunStatus.WAITING and failed[0].attempt_of("research") == ____

## The fan-in counter, by hand

Children finish in any order, and their completion notices have at-least-once delivery. The parent keeps a counter. The write that makes the counter reach `expected` wakes the aggregator.

Write the rule that the store applies atomically (`InMemoryStateStore.record_child_result` under a lock, Firestore in a transaction). Record each child **one time**. When the counter gets to `expected`, report *all done* for every notice, and for a late duplicate too. The wake-up of that duplicate is a named, de-duplicated task, so another try causes no harm. If the first wake-up did not arrive and the store answers *not done*, it is possible that the parent stays stuck.

In [ ]:
# TODO: fill in the blanks: not_seen, all_done
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: not_seen, all_done")
def record(fan, child_key, result):
    """fan = {"expected": n, "completed": 0, "results": {}}. Returns True iff every child is in."""
    if ____:
        fan["results"][child_key] = result
        fan["completed"] += 1
    return ____

fan = {"expected": 3, "completed": 0, "results": {}}
notices = [("a", 1), ("b", 2), ("a", 1), ("c", 3), ("b", 2)]          # "a" twice, and a late duplicate of "b"
seen = [record(fan, k, v) for k, v in notices]
print(seen, fan)
assert seen == [False, False, False, True, True] and fan["completed"] == 3 and fan["results"] == {"a": 1, "b": 2, "c": 3}

## Saga: undo in reverse order

`procurement` has three steps: reserve stock, charge, and book the shipment. The shipment fails after the first two steps succeeded. The engine then runs their compensations in reverse order, and it uses the stored effect records.

In [ ]:
# TODO: fill in the blanks: expected_calls
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: expected_calls")
from lra.examples.procurement_saga import ExternalSystems
ExternalSystems.reset()
engine, runner, clock, store, queue, bus = harness()
run = engine.start("procurement", {"sku": "GPU", "qty": 1, "amount": 25000, "flaky_at": "charge_payment", "fail_at": "book_shipment"})
runner.run_until_idle()
r = store.get(run.run_id); show(r); print(r.error)
print("external calls:", [c[0] for c in ExternalSystems.calls])
print("history:", [(h.step, h.kind, h.status) for h in r.history])
assert r.status == RunStatus.COMPENSATED
assert [c[0] for c in ExternalSystems.calls] == ____

## Write your own compensable step

`compensating(effect_key, undo)` makes an idempotent compensation that reads the effect record.

In [ ]:
# TODO: fill in the blanks: effect_key
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: effect_key")
from lra.patterns.saga import compensating
CALLS = []
wf = Workflow("ticketing")

@wf.step(start=True, compensate=compensating(____, lambda ctx, rec: CALLS.append(("close", rec["ticket_id"]))))
def open_ticket(ctx):
    rec = ctx.effect(____, lambda: CALLS.append(("open", "T-1")) or {"ticket_id": "T-1"})
    ctx.state["ticket"] = rec["ticket_id"]
    return Next("assign")

@wf.step(max_attempts=1)
def assign(ctx):
    raise StepFailed("no engineer available in region")     # business failure -> compensate

engine.registry.register(wf)
run = engine.start("ticketing"); runner.run_until_idle()
r = store.get(run.run_id); show(r); print(CALLS)
assert r.status == RunStatus.COMPENSATED and CALLS == [("open", "T-1"), ("close", "T-1")]

## Reflection loop exits, and the budget failing closed

Make the critic never satisfied. The loop must then exit on `max iterations`. After that, give the run a small step budget. The run must fail *before* another model call.

In [ ]:
# TODO: fill in the blanks: iterations, max_steps
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: iterations, max_steps")
engine, runner, clock, store, queue, bus = harness(routes=research_routes(first_score=3, second_score=3))
run = engine.start("research_pipeline", {"goal": "stubborn"}); runner.run_until_idle()
loop = store.get(run.run_id).state["reflect_loop"]
print(loop["exit_reason"], "| scores:", [h["score"] for h in loop["history"]])
assert loop["iteration"] == ____

engine, runner, clock, store, queue, bus = harness(routes=research_routes(first_score=1, second_score=1))
run = engine.start("research_pipeline", {"goal": "runaway"}, budget=Budget(max_steps=____, max_cost_usd=10))
runner.run_until_idle()
r = store.get(run.run_id); show(r); print(r.error)
calls_before = len(engine.llm.calls); runner.run_until_idle()
assert r.status == RunStatus.FAILED and "step budget" in r.error and len(engine.llm.calls) == calls_before

## Deadlines are absolute

A run that waited beyond its deadline must not publish when the approval arrives at last.

In [ ]:
# TODO: fill in the blanks: hours
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: hours")
engine, runner, clock, store, queue, bus = harness()
run = engine.start("research_pipeline", {"goal": "late"}, budget=Budget(deadline=clock.now() + timedelta(hours=1)))
runner.run_until_idle()
clock.advance(hours=____)
engine.resume(Event(run_id=run.run_id, key=f"editor:{run.run_id}", payload={"decision": "approve", "by": "ed"}))
runner.run_until_idle()
r = store.get(run.run_id); show(r); print(r.error)
assert r.status == RunStatus.FAILED and "publish" not in r.completed_steps

## Takeaways

* A fan-out is a set of child runs. A fan-in is an atomic counter on the parent. Partial failure is data.
* Sagas need the effect records, not a new calculation. Compensations must be idempotent.
* Every loop has three exits. Budgets and deadlines are a gate for the *next* model call, not for the current one.